### Función auxiliar para registrar metadatos y logs

In [0]:
%run "./00_utils_logging_functions"

In [0]:
from  pyspark.sql.functions import col,trim,when



spark.sql("USE dbestudiantes")


layer = "Silver"

### Funciones utiles para el proyecto

### Students – limpieza y calidad

In [0]:
from pyspark.sql.functions import col, trim, when

job_name = "clean_students"
tabla_studientes = "workspace.dbestudiantes.silver_students"

try:
    df_raw_students = spark.table("workspace.dbestudiantes.bronze_raw_students")

    #Detectar el ultimo id_ingestion de la tabla silver_students
    if spark.catalog.tableExists(tabla_studientes):
        df_students = spark.table(tabla_studientes)
        ultimo_id_ingestion = df_students.agg({"id_ingestion": "max"}).collect()[0][0]
      
    else:
        ultimo_id_ingestion = 0
        df_students = None
    
    #Filtro para obtener los nuevos registros
    df_new_students = df_raw_students.filter(col("id_ingestion") > ultimo_id_ingestion)
    
    #Transformacion de los datos
    df_clean_students = (
        df_new_students
        .withColumn("nombre",trim(col("nombre")))
        .withColumn("apellido",trim(col("apellido")))
        .withColumn("repitente",when(col("repitente")=="si",True).otherwise(False))
        .withColumn("alumno_ayudante",when(col("alumno_ayudante")=="si",True).otherwise(False))
        .filter(col("edad").between(15,30)) #calidad de datos
        
    )

    row_count = df_clean_students.count()

    #Escribir en la tabla silver_students
    if df_students is None:
        df_clean_students.write.format("delta").mode("overwrite").saveAsTable(tabla_studientes)
    else:
        df_clean_students.write.format("delta").mode("append").saveAsTable(tabla_studientes)
    
    #Registrar metadatos
    if row_count > 0:
        ultimo_id_ingestion = df_clean_students.agg({"id_ingestion": "max"}).collect()[0][0]
      
    else:
        ultimo_id_ingestion = ultimo_id_ingestion

    log_meta(job_name, layer, row_count,ultimo_id_ingestion, "SUCCESS")
    log_event(job_name, layer, "INFO", "Limpieza de la tabla STUDENTS completada correctamente.")

except Exception as e:
    log_event(job_name, layer, "ERROR", f"Fallo en CLEAN students: {str(e)}")
    log_meta(job_name, layer, -1, ultimo_id_ingestion, "ERROR")
    raise

 

### Enrollments – tipos y calidad

In [0]:
from pyspark.sql.functions import col, trim, when

job_name = "clean_enrollments"
tabla_studientes = "workspace.dbestudiantes.silver_enrollments"

try:
    df_raw_enrollments = spark.table("workspace.dbestudiantes.bronze_raw_enrollments")

    #Detectar el ultimo id_ingestion de la tabla silver_enrollments
    if spark.catalog.tableExists(tabla_studientes):
        df_enrollments = spark.table(tabla_studientes)
        ultimo_id_ingestion = df_enrollments.agg({"id_ingestion": "max"}).collect()[0][0]
      
    else:
        ultimo_id_ingestion = 0
        df_enrollments = None
    
    #Filtro para obtener los nuevos registros
    df_new_enrollments = df_raw_enrollments.filter(col("id_ingestion") > ultimo_id_ingestion)
    
    #Transformacion de los datos
    df_clean_enrollments = (
        df_new_enrollments
        .withColumn("fecha",col("fecha").cast("date"))
        .filter(col("nota").between(0,10)) #calidad de datos
        
    )

    row_count = df_clean_enrollments.count()

    #Escribir en la tabla silver_enrollments
    if df_enrollments is None:
        df_clean_enrollments.write.format("delta").mode("overwrite").saveAsTable(tabla_studientes)
    else:
        df_clean_enrollments.write.format("delta").mode("append").saveAsTable(tabla_studientes)
    
    #Registrar metadatos
    if row_count > 0:
        ultimo_id_ingestion = df_clean_enrollments.agg({"id_ingestion": "max"}).collect()[0][0]
      
    else:
        ultimo_id_ingestion = ultimo_id_ingestion

    log_meta(job_name, layer, row_count,ultimo_id_ingestion, "SUCCESS")
    log_event(job_name, layer, "INFO", "Limpieza de la tabla ENROLLMENTS completada correctamente.")

except Exception as e:
    log_event(job_name, layer, "ERROR", f"Fallo en CLEAN enrollments: {str(e)}")
    log_meta(job_name, layer, -1, ultimo_id_ingestion, "ERROR")
    raise e

  


### Subjects – directo a clean

In [0]:


from pyspark.sql.functions import col, trim, when

job_name = "clean_subjects"
tabla_subjects = "workspace.dbestudiantes.silver_subjects"

try:
    df_raw_subjects = spark.table("workspace.dbestudiantes.bronze_raw_subjects")

    #Detectar el ultimo id_ingestion de la tabla silver_subjects
    if spark.catalog.tableExists(tabla_subjects):
        df_subjects = spark.table(tabla_subjects)
        ultimo_id_ingestion = df_subjects.agg({"id_ingestion": "max"}).collect()[0][0]
      
    else:
        ultimo_id_ingestion = 0
        df_subjects = None
    
    #Filtro para obtener los nuevos registros
    df_new_subjects = df_raw_subjects.filter(col("id_ingestion") > ultimo_id_ingestion)
    
    #Transformacion de los datos
    df_clean_subjects = (
        df_new_subjects
        
    )

    row_count = df_clean_subjects.count()

     #Escribir en la tabla silver_enrollments
    if df_enrollments is None:
        df_clean_enrollments.write.format("delta").mode("overwrite").saveAsTable(tabla_subjects)
    else:
        df_clean_enrollments.write.format("delta").mode("append").saveAsTable(tabla_subjects)

    
    #Registrar metadatos
    if row_count > 0:
        ultimo_id_ingestion = df_clean_subjects.agg({"id_ingestion": "max"}).collect()[0][0]
      
    else:
        ultimo_id_ingestion = ultimo_id_ingestion

    log_meta(job_name, layer, row_count,ultimo_id_ingestion, "SUCCESS")
    log_event(job_name, layer, "INFO", "Limpieza de la tabla SUBJECTS completada correctamente.")

except Exception as e:
    log_event(job_name, layer, "ERROR", f"Fallo en CLEAN subjects: {str(e)}")
    log_meta(job_name, layer, -1, ultimo_id_ingestion, "ERROR")
    raise e 

   